# HW 2 - Разложение матриц градиентным методом


Цель задания: В ходе реализации [разложения Таккера](https://proceedings.neurips.cc/paper/2018/file/45a766fa266ea2ebeb6680fa139d2a3d-Paper.pdf) градиентным методом освоить pyTorch и реализовать подходы оптимизации параметров модели (в отсутствии готовых решений).


[Более-менее внятное описание алгоритма канонического разложения](https://www.alexejgossmann.com/tensor_decomposition_tucker/) - само аналитическое разложение вам реализовывать НЕ НУЖНО


In [1]:
import time
import numpy as np
import torch
import tensorly as tl
from torch import nn
from torch.optim.optimizer import Optimizer


torch.manual_seed(0)

## 1 Создайте 3х мерный тензор

Размер тензора не меньше 100 по каждой из размерностей.

Заполните случайными целыми числами в диапазоне от 0 до 9.


Примечание: разложение будет корректно работать со случайным тензором, только если изначально создавать случайные ядро и матрицы, а потом по ним формировать тензор. Работайте с типом _torch.Tensor.double_.


In [2]:
def get_tensor(size=(100, 200, 150), r=10):
  U = []
  for dim in size:
    U_i = torch.randn(dim, r, dtype=torch.double)
    U_i, _ = torch.linalg.qr(U_i)
    U.append(U_i)

  G = torch.randn(r, r, r, dtype=torch.double) * 0.1
  data = torch.einsum("ijk,ai,bj,ck->abc", G, *U)

  return data, U, G

Сгенерируйте тензор и добавьте к нему случайный шум с размерностью _1e-2_


In [3]:
size = (100, 200, 300)
r = 10

data, U, G = get_tensor(size, r)
data.shape, [u.shape for u in U], G.shape

(torch.Size([100, 200, 300]),
 [torch.Size([100, 10]), torch.Size([200, 10]), torch.Size([300, 10])],
 torch.Size([10, 10, 10]))

In [4]:
noise = torch.randn_like(data) * 1e-2
noisy_data = data + noise

print(f"Нормы: исходный тензор = {torch.norm(data):.4f}, шум = {torch.norm(noise):.4f}")

Нормы: исходный тензор = 3.2520, шум = 24.4976


Вопрос:
Почему задание не имеет смысла для полностью случайного тензора и зачем добавлять шум? _не отвечать нельзя_


Ответ:

Задание не имеет смысла для полностью случайного тензора, потому что:

1. Полностью случайный тензор имеет максимальный ранг и плохо сжимается
2. Разложение Такера эффективно для тензоров с структурой низкого ранга
3. Шум помогает проверить устойчивость алгоритма к небольшим возмущениям


## 2 Реализуйте метод для восстановления тензора по разложению


In [5]:
# Функция, восстанавливающая тензор по ядру и матрицам
def repair_tensor(G_, U):
  # data - восстановленный тензор из матриц и ядра
  # U - список матриц
  # G_ - ядро разложения

  U1, U2, U3 = U

  temp = G_.clone()
  temp = torch.tensordot(U1, temp, dims=([1], [0]))
  temp = torch.tensordot(U2, temp, dims=([1], [1]))
  temp = temp.permute(1, 0, 2)

  temp = torch.tensordot(U3, temp, dims=([1], [2]))
  temp = temp.permute(1, 2, 0)
  return temp

In [6]:
# Проверим восстановление
reconstructed = repair_tensor(G, U)
mse_original = torch.mean((data - reconstructed) ** 2)
print(f"\nMSE между исходным и восстановленным: {mse_original:.2e}")


MSE между исходным и восстановленным: 0.00e+00


## 3 Сделайте разложение библиотечным методом

Пакет можете брать любой


In [7]:
from tensorly.decomposition import tucker


In [8]:
tl.set_backend("pytorch")

start_time = time.time()
core, factors = tucker(noisy_data, rank=[r, r, r], init="random", n_iter_max=100)
library_time = time.time() - start_time

# Восстанавливаем тензор из библиотечного разложения
library_reconstructed = tl.tucker_to_tensor((core, factors))
mse_library = torch.mean((noisy_data - library_reconstructed) ** 2)

print(f"Время библиотечного разложения: {library_time:.2f} сек")
print(f"MSE библиотечного разложения: {mse_library:.2e}")
print(f"Форма ядра: {core.shape}")
print(f"Формы факторов: {[f.shape for f in factors]}")


Время библиотечного разложения: 0.21 сек
MSE библиотечного разложения: 9.99e-05
Форма ядра: torch.Size([10, 10, 10])
Формы факторов: [torch.Size([100, 10]), torch.Size([200, 10]), torch.Size([300, 10])]


## 4 Реализуйте разложение градиентным методом


### 4.1 Реализуйте _optimizer_

Можно взять из исходников _PyTorch_ и отнаследоваться от _torch.optim.optimizer_.
Используйте квадратичный _Loss_.


In [9]:
class Opt(Optimizer):
  def __init__(self, params, lr=1e-3):
    defaults = dict(lr=lr)
    super(Opt, self).__init__(params, defaults)

  def step(self, closure=None):
    loss = None
    if closure is not None:
      loss = closure()

    for group in self.param_groups:
      learning_rate = group["lr"]

      for param in group["params"]:
        if param.grad is None:
          continue

        param.data.add_(param.grad.data, alpha=-learning_rate)

    return loss

### 4.2 Реализуйте цикл оптимизации параметров


Стоит параметры оптимизировать сразу на GPU


In [10]:
def init_params(dimensions, rank):
  U = []
  for i in range(3):
    size_i, r_i = dimensions[i], rank
    std = torch.sqrt(torch.tensor(2.0 / (size_i + r_i)))
    U.append(
      nn.Parameter(torch.randn(size_i, r_i, dtype=torch.double, device="cpu") * std)
    )
  G = nn.Parameter(torch.randn(rank, rank, rank, dtype=torch.double, device="cpu") * 0.1)
  return G, U

In [11]:
size = (100, 200, 150)
r = 10

data, U, G = get_tensor(size, r)
noise = torch.randn_like(data) * 0.01
noisy_data = data + noise

print(f"Исходный тензор: [{data.min().item():.2f}, {data.max().item():.2f}]")
print(f"Зашумленный: [{noisy_data.min().item():.2f}, {noisy_data.max().item():.2f}]")

Исходный тензор: [-0.02, 0.02]
Зашумленный: [-0.05, 0.05]


In [12]:
def calculate_mse(original, reconstructed):
  mse = torch.mean((original - reconstructed) ** 2)
  return mse

In [13]:
G_est, U_est = init_params(size, r)
optimizer = torch.optim.Adam([G_est] + U_est, lr=0.01)

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.9)

epochs = 1000

prev_loss = 0.0

for epoch in range(epochs):
  optimizer.zero_grad()
  reconstructed = repair_tensor(G_est, U_est)
  loss = calculate_mse(reconstructed, noisy_data)
  loss.backward()

  torch.nn.utils.clip_grad_norm_([G_est] + U_est, max_norm=1.0)
  optimizer.step()
  scheduler.step()

  curr_loss = loss.item()

  if epoch > 0:
    loss_change = abs(prev_loss - curr_loss)
    if loss_change < 1e-10:
      print(f"Остановка на эпохе {epoch}: изменение loss = {loss_change:.2e}")
      best_G = G_est.detach().clone()
      best_U = [u.detach().clone() for u in U_est]
      break

  prev_loss = curr_loss
  best_G = G_est.detach().clone()
  best_U = [u.detach().clone() for u in U_est]

  if epoch % 50 == 0:
    current_lr = optimizer.param_groups[0]["lr"]
    print(f"Эпоха {epoch}: Loss = {curr_loss:.8f}, LR = {current_lr:.6f}")


G_est.data = best_G.data
for i, u in enumerate(U_est):
  u.data = best_U[i].data

with torch.no_grad():
  reconstructed_final = repair_tensor(G_est, U_est)
  mse_noisy = calculate_mse(reconstructed_final, noisy_data)
  mse_clean = calculate_mse(reconstructed_final, data)

print(f"\nРезультаты:")
print(f"MSE (зашумленный): {mse_noisy.item():.8f}")
print(f"MSE (исходный): {mse_clean.item():.8f}")

Эпоха 0: Loss = 0.00012435, LR = 0.010000
Эпоха 50: Loss = 0.00010321, LR = 0.010000
Эпоха 100: Loss = 0.00010311, LR = 0.009000
Эпоха 150: Loss = 0.00010277, LR = 0.009000
Эпоха 200: Loss = 0.00010172, LR = 0.008100
Эпоха 250: Loss = 0.00010006, LR = 0.008100
Эпоха 300: Loss = 0.00009976, LR = 0.007290
Остановка на эпохе 301: изменение loss = 9.58e-11

Результаты:
MSE (зашумленный): 0.00009976
MSE (исходный): 0.00000018


## 5 Приведите сравнение скорости работы и ошибки восстановления методом из пакета и реализованного градиентного

Сравнение может считаться ± объективным с размером выборки от 10.


In [14]:
def compare_methods(num_experiments=10):
  gd_times = []
  gd_errors = []
  library_times = []
  library_errors = []

  tl.set_backend("numpy")

  for exp in range(num_experiments):
    print(f"Сравнение № {exp + 1}")

    size = (100, 200, 150)
    r = 10
    data, U_true, G_true = get_tensor(size, r)
    noise = torch.randn_like(data) * 1e-2
    noisy_data = data + noise

    # Градиентный метод
    start_time = time.time()

    G_est, U_est = init_params(size, r)
    optimizer = torch.optim.Adam([G_est] + U_est, lr=0.01)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=100, gamma=0.9)

    epochs = 1000
    prev_loss = 0.0
    best_G = None
    best_U = None

    for epoch in range(epochs):
      optimizer.zero_grad()
      reconstructed = repair_tensor(G_est, U_est)
      loss = calculate_mse(reconstructed, noisy_data)
      loss.backward()

      torch.nn.utils.clip_grad_norm_([G_est] + U_est, max_norm=1.0)
      optimizer.step()
      scheduler.step()

      curr_loss = loss.item()

      if epoch > 0:
        loss_change = abs(prev_loss - curr_loss)
        if loss_change < 1e-10:
          print(f"  Остановка на эпохе {epoch}: изменение loss = {loss_change:.2e}")
          best_G = G_est.detach().clone()
          best_U = [u.detach().clone() for u in U_est]
          break

      prev_loss = curr_loss
      best_G = G_est.detach().clone()
      best_U = [u.detach().clone() for u in U_est]

    if best_G is not None:
      G_est.data = best_G.data
      for i, u in enumerate(U_est):
        u.data = best_U[i].data

    with torch.no_grad():
      reconstructed_gd = repair_tensor(G_est, U_est)
      gd_error = calculate_mse(reconstructed_gd, data).item()

    gd_time = time.time() - start_time
    gd_times.append(gd_time)
    gd_errors.append(gd_error)
    print(f"  Градиентный метод: время={gd_time:.2f}с, MSE={gd_error:.8f}")

    # Библиотечный метод
    start_time = time.time()

    tensor_np = noisy_data.detach().numpy()

    core, factors = tl.decomposition.tucker(tensor_np, rank=[r, r, r])
    reconstructed_lib = tl.tucker_to_tensor((core, factors))
    reconstructed_lib = torch.from_numpy(reconstructed_lib).double()

    lib_time = time.time() - start_time
    library_times.append(lib_time)
    lib_error = calculate_mse(reconstructed_lib, data).item()
    library_errors.append(lib_error)
    print(f"  Библиотечный метод: время={lib_time:.2f}с, MSE={lib_error:.8f}")
    print()

  print("=" * 50)
  print("ИТОГОВЫЕ РЕЗУЛЬТАТЫ:")
  print("=" * 50)

  print("Градиентный метод:")
  print(f"  Среднее время: {np.mean(gd_times):.2f} ± {np.std(gd_times):.2f} с")
  print(f"  Средняя ошибка: {np.mean(gd_errors):.2e} ± {np.std(gd_errors):.2e}")

  print("Библиотечный метод:")
  print(f"  Среднее время: {np.mean(library_times):.2f} ± {np.std(library_times):.2f} с")
  print(f"  Средняя ошибка: {np.mean(library_errors):.2e} ± {np.std(library_errors):.2e}")
  print()


# Запуск сравнения
print("Начало сравнения методов...")
results = compare_methods(num_experiments=10)

Начало сравнения методов...
Сравнение № 1
  Остановка на эпохе 324: изменение loss = 9.56e-11
  Градиентный метод: время=11.09с, MSE=0.00000018
  Библиотечный метод: время=0.92с, MSE=0.00000018

Сравнение № 2
  Остановка на эпохе 323: изменение loss = 9.49e-11
  Градиентный метод: время=11.64с, MSE=0.00000018
  Библиотечный метод: время=0.90с, MSE=0.00000018

Сравнение № 3
  Остановка на эпохе 355: изменение loss = 9.82e-11
  Градиентный метод: время=12.79с, MSE=0.00000019
  Библиотечный метод: время=0.94с, MSE=0.00000019

Сравнение № 4
  Остановка на эпохе 322: изменение loss = 9.95e-11
  Градиентный метод: время=9.51с, MSE=0.00000018
  Библиотечный метод: время=0.71с, MSE=0.00000018

Сравнение № 5
  Остановка на эпохе 335: изменение loss = 9.19e-11
  Градиентный метод: время=9.99с, MSE=0.00000018
  Библиотечный метод: время=0.70с, MSE=0.00000018

Сравнение № 6
  Остановка на эпохе 383: изменение loss = 9.86e-11
  Градиентный метод: время=11.29с, MSE=0.00000019
  Библиотечный метод: в